In [2]:
import os
from getpass import getpass
from pathlib import Path

import requests
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)

In [16]:
from api_key import key 
key = key.strip()

In [20]:
SYMBOL = "IBM"  # Можешь заменить на другой тикер, например AAPL
API_KEY = key


BASE_URL = "https://www.alphavantage.co/query"
PARAMS = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "compact",  # Последние 100 торговых сессий
    "apikey": API_KEY,
}

In [21]:
response = requests.get(BASE_URL, params=PARAMS, timeout=30)
response.raise_for_status()
payload = response.json()

for error_key in ("Error Message", "Note", "Information"):
    if error_key in payload:
        raise RuntimeError(f"Ответ API ({error_key}): {payload[error_key]}")

daily = payload.get("Time Series (Daily)")
if not isinstance(daily, dict) or not daily:
    raise RuntimeError(f"Не найдены дневные котировки. Ключи ответа: {list(payload)}")

print(f"API ответил: получено {len(daily)} торговых сессий для {SYMBOL}")

API ответил: получено 100 торговых сессий для IBM


In [22]:
df = (
    pd.DataFrame.from_dict(daily, orient="index")
    .rename_axis("date")
    .reset_index()
    .rename(columns={
        "1. open": "open",
        "2. high": "high",
        "3. low": "low",
        "4. close": "close",
        "5. volume": "volume",
    })
)
display(df.head())

,date,open,high,low,close,volume
0,2026-11-11,244.7200,249.3100,242.8800,248.2200,3419582
1,2026-11-10,244.3800,246.0000,242.7000,244.5300,2971839
2,2026-11-09,246.0000,248.8100,245.1957,246.1100,4629816
3,2026-11-06,242.9200,246.3400,240.3500,245.0000,3494073
4,2026-11-05,255.9200,256.1750,245.7200,247.1100,5339546


In [23]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
numeric_cols = ["open", "high", "low", "close", "volume"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)
print("Размер таблицы:", df.shape)
display(df.head())

Размер таблицы: (100, 6)


,date,open,high,low,close,volume
0,2026-06-23,261.580,267.5300,255.2600,264.94,15686117
1,2026-06-24,261.843,265.0700,256.1836,262.96,8149935
2,2026-06-25,267.670,268.7559,256.0000,258.27,8539694
3,2026-06-26,258.940,273.1350,258.2800,271.63,9719927
4,2026-06-29,274.300,278.1500,269.0600,278.00,6464204


In [24]:
print("Число дубликатов дат:", df["date"].duplicated().sum())
quality = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(),
})
display(quality)

assert df["date"].notna().all(), "В датах обнаружены пропуски"
assert not df["date"].duplicated().any(), "Есть повторяющиеся даты"
assert df[numeric_cols].notna().all().all(), "В котировках есть пропуски"
assert (df[["open", "high", "low", "close"]] > 0).all().all(), "Цена <= 0"
assert (df["volume"] >= 0).all(), "Объем торговли < 0"
assert (df["high"] >= df[["open", "low", "close"]].max(axis=1)).all(), "Проверь high"
assert (df["low"] <= df[["open", "high", "close"]].min(axis=1)).all(), "Проверь low"

Число дубликатов дат: 0


,data_type,missing,unique_values
date,datetime64[ns],0,100
open,float64,0,100
high,float64,0,100
low,float64,0,98
close,float64,0,100
volume,int64,0,100


In [25]:
df.info()
display(df[["open", "close", "volume"]].describe().round(2))
print("Период:", df["date"].min().date(), "—", df["date"].max().date())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    100 non-null    datetime64[ns]
 1   open    100 non-null    float64       
 2   high    100 non-null    float64       
 3   low     100 non-null    float64       
 4   close   100 non-null    float64       
 5   volume  100 non-null    int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 4.8 KB


,open,close,volume
count,100.00,100.00,100.00
mean,239.31,240.28,7404181.23
std,20.72,21.62,7278710.08
min,202.64,205.77,2764896.00
25%,227.37,227.05,4344085.00
50%,234.38,234.80,5645844.50
75%,246.76,249.33,8162472.75
max,305.66,306.13,67440873.00


Период: 2026-06-23 — 2026-11-11


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["open"], bins=15, edgecolor="white")
axes[0].set_title("Распределение цены открытия")
axes[0].set_xlabel("Open, USD")
axes[0].set_ylabel("Число торговых дней")

axes[1].hist(df["close"], bins=15, edgecolor="white")
axes[1].set_title("Распределение цены закрытия")
axes[1].set_xlabel("Close, USD")
axes[1].set_ylabel("Число торговых дней")

plt.tight_layout()
plt.show()